# Playground — Step 7: Evaluation Harness

`frames.evaluation.harness` provides:

- **`EvaluationHarness`** — decoupled from generation: hand it any (prompts, texts) plus a
  config dict, and it computes per-concept **presence success** (WordNet member lemmas in
  the continuation), **perplexity** vs an unsteered greedy baseline with a **fluency
  guardrail flag** (default ratio > 2.5), and appends one JSONL record per generation.
- **`RecordingScorer`** — wraps any scorer/scheduler and captures per-step score traces
  (per-concept + aggregated, for each input's representative beam).

Gate for this step: run on the golden prompts, JSONL parses, PPL numbers plausible.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root, so `frames` imports work

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd
import torch

from frames.evaluation.harness import EvaluationHarness, RecordingScorer
from frames.representations import FrameUnembeddingRepresentation, aggregators
from frames.representations.schedulers import RoundRobin

In [ ]:
fur = FrameUnembeddingRepresentation.from_model_id(
    "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
    device_map="cuda:0",
    torch_dtype=torch.float16,
)

MIN_LEMMAS = 11
MAX_TOKENS = 3
K = 3
STEPS = 16

# the golden prompts — per the step's gate (works from repo root and playground/)
ROOT = Path.cwd() if (Path.cwd() / "tests").exists() else Path.cwd().parent
GOLDEN = json.loads((ROOT / "tests/golden/single_concept.json").read_text())
PROMPTS = GOLDEN["prompts"]
SYNSETS = ["joy.n.01", "dog.n.01"]

concepts = [fur.get_concept_cached(s, MIN_LEMMAS, MAX_TOKENS) for s in SYNSETS]

LOG_PATH = ROOT / "playground" / "cache" / "step07_eval.jsonl"
LOG_PATH.unlink(missing_ok=True)  # fresh log for this session

harness = EvaluationHarness(
    fur, LOG_PATH, min_lemmas_per_synset=MIN_LEMMAS, max_token_count=MAX_TOKENS
)

## 1 — Baseline once, reuse everywhere

The unsteered greedy continuations anchor the fluency guardrail for all methods.

In [ ]:
baseline_texts = harness.generate_baseline(PROMPTS, max_new_tokens=STEPS + 2)
print(baseline_texts[0][-120:])

## 2 — Evaluate three methods on the golden prompts

F2.a (z-scored weighted sum, with per-step trace recording), F1.a (mean frame),
and F3.a (round-robin).

In [ ]:
recorder = RecordingScorer(aggregators.weighted_sum)
texts_f2, probe_f2 = fur.generate_with_topk_multi_guide(
    PROMPTS, guides=concepts, k=K, steps=STEPS, normalize="zscore", scorer=recorder,
)
records_f2 = harness.evaluate(
    PROMPTS, texts_f2, SYNSETS,
    config={"family": "F2.a", "normalize": "zscore", "k": K, "steps": STEPS},
    probe=probe_f2, recorder=recorder, baseline_texts=baseline_texts,
)

In [ ]:
texts_f1, probe_f1 = fur.quick_generate_with_topk_mean_frame_guide(
    PROMPTS, guides=[[s] for s in SYNSETS],
    min_lemmas_per_synset=MIN_LEMMAS, max_token_count=MAX_TOKENS, k=K, steps=STEPS,
)
records_f1 = harness.evaluate(
    PROMPTS, texts_f1, SYNSETS,
    config={"family": "F1.a", "k": K, "steps": STEPS},
    probe=probe_f1, baseline_texts=baseline_texts,
)

texts_f3, probe_f3 = fur.generate_with_topk_multi_guide(
    PROMPTS, guides=concepts, k=K, steps=STEPS, scorer=RoundRobin(),
)
records_f3 = harness.evaluate(
    PROMPTS, texts_f3, SYNSETS,
    config={"family": "F3.a", "k": K, "steps": STEPS},
    probe=probe_f3, baseline_texts=baseline_texts,
)

## 3 — Gate checks: JSONL parses, PPL plausible

In [ ]:
lines = LOG_PATH.read_text().strip().split("\n")
parsed = [json.loads(line) for line in lines]
assert len(parsed) == 3 * len(PROMPTS), "one record per prompt per method"
assert all(0.5 < r["ppl"] < 1000 for r in parsed), "PPL out of plausible range"
assert all(0.5 < r["baseline_ppl"] < 1000 for r in parsed)
print(f"gate passed: {len(parsed)} records parse, PPL in range")
print(f"log: {LOG_PATH}")

## 4 — Summary table

In [ ]:
rows = []
for record in parsed:
    rows.append(
        {
            "family": record["config"]["family"],
            "joy": record["success"]["joy.n.01"],
            "dog": record["success"]["dog.n.01"],
            "both": all(record["success"].values()),
            "ppl_ratio": record["ppl_ratio"],
            "flagged": record["fluency_flag"],
        }
    )
table = pd.DataFrame(rows)
summary = table.groupby("family").agg(
    joy_rate=("joy", "mean"),
    dog_rate=("dog", "mean"),
    both_rate=("both", "mean"),
    mean_ppl_ratio=("ppl_ratio", "mean"),
    flags=("flagged", "sum"),
)
summary.round(3)

## 5 — Per-step trace (from the RecordingScorer)

Normalized per-concept scores of the representative beam at each step, first prompt.

In [ ]:
trace = records_f2[0]["steps"]
per_concept = torch.tensor([t["per_concept"] for t in trace])

plt.figure(figsize=(8, 3))
for c, synset in enumerate(SYNSETS):
    plt.plot(per_concept[:, c], label=synset, marker="o", markersize=3)
plt.xlabel("generation step")
plt.ylabel("z-scored projection")
plt.title("per-step concept scores (F2.a, prompt 0)")
plt.legend()
plt.tight_layout()
plt.show()